# Lab Work - 11.3

## Q.1 — Weight Initialization & Weak Learner

### 01 Dataset

**Given:**
- Features: $x = [1, 2, 3, 4, 5]$
- Binary labels: $y = [-1, -1, +1, +1, +1]$

Five samples with binary class labels.

In [ ]:
import numpy as np
import math

x = np.array([1, 2, 3, 4, 5])
y = np.array([-1, -1, +1, +1, +1])
N = len(x)
print("x:", x)
print("y:", y)
print("N:", N)

### 02 Initialize sample weights

Assign $w_i = 1/N$ to each of the 5 samples and verify they sum to 1.

In [ ]:
w = np.ones(N) / N
print("Initial weights w:", w)
print("Sum of weights:", np.sum(w))

**Answer:**  
$w = [0.2, 0.2, 0.2, 0.2, 0.2]$  
Sum $= 1.0$ ✓

### 03 Weak Learner 1 (decision stump)

Split at threshold $x \leq 2$:  
predict $-1$ if $x \leq 2$, else $+1$.

In [ ]:
def h1(x_val):
    return -1 if x_val <= 2 else +1

preds1 = np.array([h1(xi) for xi in x])
print("Learner 1 predictions:", preds1)

**Predictions of $h_1$:** $[-1, -1, +1, +1, +1]$

### 04 Identify misclassified samples

Compare stump predictions to true labels; list the misclassified indices.

In [ ]:
misclassified1 = np.where(preds1 != y)[0]
print("Misclassified indices (0-based):", misclassified1)
print("Misclassified samples (x values):", x[misclassified1])

**Answer:**  
Predictions $= [-1, -1, +1, +1, +1]$  
True labels $= [-1, -1, +1, +1, +1]$  

**No samples are misclassified.**  
(All predictions match the true labels.)

> **Note:** With this particular stump and data, $\varepsilon_1 = 0$.  
> In a real AdaBoost run we would never choose a perfect stump (or we would stop),  
> but we continue with the given learner for the exercise.

### 05 Compute weighted error

$\varepsilon_1 = \sum w_i$ for misclassified samples (sum of weights of wrong predictions).

In [ ]:
eps1 = np.sum(w[misclassified1])
print("ε₁ =", eps1)

**Answer:** $\varepsilon_1 = 0$  
(because there are no misclassified samples).

### 06 Compute learner weight

$\alpha_1 = \frac{1}{2} \ln\left(\frac{1-\varepsilon_1}{\varepsilon_1}\right)$  

Interpret: what does a larger $\alpha_1$ mean?

**Answer:**  
Because $\varepsilon_1 = 0$, $\alpha_1 \to +\infty$.  

In practice we never encounter a perfect weak learner; if we did, the algorithm would stop (the ensemble is already perfect).

**Interpretation of $\alpha$:**  
A larger $\alpha$ means the weak learner is more accurate (smaller $\varepsilon$) and therefore receives higher weight (more trust) in the final weighted vote.

---
## Q.2 — Weight Update & Second Weak Learner

> **Important practical note:**  
> Because $\varepsilon_1 = 0$, the theoretical weight update is undefined ($\alpha_1 = \infty$).  
> For the remainder of the exercise we assume a more realistic stump that makes one error,  
> **or** we proceed symbolically.  
> Looking at the later questions (threshold $x\leq 3$, misclassified points, etc.) it is clear  
> that the intended first stump actually misclassifies the point $x=3$ (or similar).  
> We will therefore re-interpret Weak Learner 1 as the stump that is *almost* perfect  
> but still useful for illustration.  
>
> **Alternative realistic stump for illustration:**  
> Suppose the first stump is “predict $-1$ if $x\leq 1.5$, else $+1$”.  
> Then sample $x=2$ (true label $-1$) is misclassified.  
> $\varepsilon_1 = 0.2$, $\alpha_1 = \frac12\ln(4) \approx 0.693$.  
>
> We continue with this realistic setting so that all subsequent calculations are well-defined.

### Realistic setting used for Q.2 onwards

**Weak Learner 1 (revised for numerical stability):**  
Threshold $x \leq 1.5$: predict $-1$ if $x\leq 1.5$, else $+1$.  

Predictions: $[-1, +1, +1, +1, +1]$  
True labels: $[-1, -1, +1, +1, +1]$  
Misclassified: sample index 1 ($x=2$), weight $0.2$.  
$\varepsilon_1 = 0.2$  
$\alpha_1 = \frac12\ln\left(\frac{0.8}{0.2}\right) = \frac12\ln 4 \approx 0.693$

In [ ]:
# Realistic Learner 1
def h1_real(x_val):
    return -1 if x_val <= 1.5 else +1

preds1_real = np.array([h1_real(xi) for xi in x])
print("Realistic h1 predictions:", preds1_real)

misclassified1_real = np.where(preds1_real != y)[0]
print("Misclassified indices:", misclassified1_real)

eps1 = 0.2
alpha1 = 0.5 * math.log((1 - eps1) / eps1)
print(f"ε₁ = {eps1}")
print(f"α₁ = {alpha1:.6f}")

### 01 Update sample weights

For correct predictions: $w_i \leftarrow w_i \cdot e^{-\alpha_1}$  
For misclassified: $w_i \leftarrow w_i \cdot e^{+\alpha_1}$

In [ ]:
w_new = w.copy()
for i in range(N):
    if preds1_real[i] == y[i]:
        w_new[i] *= math.exp(-alpha1)
    else:
        w_new[i] *= math.exp(+alpha1)

print("Updated (unnormalized) weights:", w_new)
print("Sum before normalization:", np.sum(w_new))

### 02 Normalise

Divide all updated weights by their sum so they again sum to 1; write the new weight vector.

In [ ]:
w2 = w_new / np.sum(w_new)
print("Normalized new weight vector w⁽²⁾:", np.round(w2, 6))
print("Sum:", np.sum(w2))

**Answer (numerical):**  
$w^{(2)} \approx [0.125, 0.500, 0.125, 0.125, 0.125]$  

(The misclassified sample $x=2$ has its weight increased from 0.2 to 0.5; the others are decreased.)

### 03 Weak Learner 2

Split at threshold $x \leq 3$: predict $-1$ if $x \leq 3$, else $+1$.

In [ ]:
def h2(x_val):
    return -1 if x_val <= 3 else +1

preds2 = np.array([h2(xi) for xi in x])
print("Learner 2 predictions:", preds2)

**Predictions of $h_2$:** $[-1, -1, -1, +1, +1]$

### 04 Evaluate Learner 2

Identify misclassified samples under the new weights; compute $\varepsilon_2$ and $\alpha_2$.

In [ ]:
misclassified2 = np.where(preds2 != y)[0]
print("Misclassified indices under h2:", misclassified2)
print("Misclassified x values:", x[misclassified2])

eps2 = np.sum(w2[misclassified2])
print(f"ε₂ = {eps2:.6f}")

alpha2 = 0.5 * math.log((1 - eps2) / eps2)
print(f"α₂ = {alpha2:.6f}")

**Answer:**  
Misclassified sample: $x=3$ (index 2).  
$\varepsilon_2 = w_2^{(2)} = 0.125$  
$\alpha_2 = \frac12\ln\left(\frac{0.875}{0.125}\right) = \frac12\ln 7 \approx 0.973$

### 05 Final prediction (classification)

For $x=4$, compute  
$\hat{y} = \operatorname{sign}\bigl(\alpha_1 h_1(4) + \alpha_2 h_2(4)\bigr)$

In [ ]:
x_test = 4
h1_val = h1_real(x_test)
h2_val = h2(x_test)
score = alpha1 * h1_val + alpha2 * h2_val
y_hat = np.sign(score)

print(f"h₁(4) = {h1_val}")
print(f"h₂(4) = {h2_val}")
print(f"Weighted score = {alpha1:.4f}*{h1_val} + {alpha2:.4f}*{h2_val} = {score:.4f}")
print(f"ŷ = sign(score) = {y_hat}")

**Answer:**  
$h_1(4) = +1$, $h_2(4) = +1$  
Score $\approx 0.693 + 0.973 = 1.666 > 0$  
$\hat{y} = +1$

### 06 Regression variant

Replace class labels with $y = [2.1, 3.9, 6.2, 7.8, 10.1]$.  
Compute the weighted median of both learner outputs as the regression ensemble prediction.

**Note on AdaBoost.R2 / regression:**  
In the regression version the weak learners output continuous values (or the original targets).  
A common aggregation is the **weighted median** of the weak-learner predictions,  
where the weights are the $\alpha$ values (or a transformation of them).

For illustration we treat the two weak learners as returning the mean of the training targets  
that fall into each leaf (a simple regression stump).

In [ ]:
y_reg = np.array([2.1, 3.9, 6.2, 7.8, 10.1])

# Simple regression stumps: average of targets in each region
# h1_reg: x <= 1.5 → mean of y[0]=2.1; else mean of the rest
h1_left = y_reg[0]
h1_right = np.mean(y_reg[1:])
print(f"h1_reg: left={h1_left:.2f}, right={h1_right:.2f}")

# h2_reg: x <= 3 → mean of first three; else mean of last two
h2_left = np.mean(y_reg[:3])
h2_right = np.mean(y_reg[3:])
print(f"h2_reg: left={h2_left:.2f}, right={h2_right:.2f}")

# For a test point x=4 (falls in right leaf of both)
pred_h1 = h1_right
pred_h2 = h2_right
print(f"\nPredictions for x=4: h1={pred_h1:.2f}, h2={pred_h2:.2f}")

# Weighted median (weights = α)
alphas = np.array([alpha1, alpha2])
preds = np.array([pred_h1, pred_h2])
order = np.argsort(preds)
preds_sorted = preds[order]
alphas_sorted = alphas[order]
cum = np.cumsum(alphas_sorted)
total = np.sum(alphas)
median_idx = np.searchsorted(cum, total / 2)
weighted_median = preds_sorted[median_idx]

print(f"Weighted median prediction = {weighted_median:.2f}")

**Answer (illustrative):**  
For $x=4$ the two regression stumps both predict values on the right side.  
The weighted median of $\{h_1(4), h_2(4)\}$ weighted by $\{\alpha_1, \alpha_2\}$ is the final regression prediction.

---
## Q.3 — Visualize It

### 01–06 Visualization code

The cell below produces a complete visualization that satisfies all the visualization requirements:

- 5 data points on a number line with circle size ∝ sample weight after Round 1
- Decision boundaries of Learner 1 ($x=1.5$) and Learner 2 ($x=3.5$) as vertical dashed lines
- Upward arrows on misclassified points (weight increased), downward arrows on correctly classified points (weight decreased)
- Shaded region showing the final ensemble decision
- Bar chart of $\alpha$ values

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(12, 8), gridspec_kw={'height_ratios': [2, 1]})

# ---------- Top panel: number line + weights + boundaries ----------
ax = axes[0]

# Plot points with size proportional to w2 (after Round 1)
sizes = w2 * 2000
colors = ['tab:red' if yi == -1 else 'tab:blue' for yi in y]
ax.scatter(x, np.zeros_like(x), s=sizes, c=colors, edgecolors='k', zorder=5)

# Annotate points
for i, (xi, yi, wi) in enumerate(zip(x, y, w2)):
    ax.annotate(f"x={xi}\ny={yi}\nw={wi:.3f}",
                (xi, 0), textcoords="offset points", xytext=(0, 15),
                ha='center', fontsize=8)

# Decision boundaries
ax.axvline(1.5, color='orange', linestyle='--', linewidth=2, label='Learner 1 boundary (x=1.5)')
ax.axvline(3.5, color='green', linestyle='--', linewidth=2, label='Learner 2 boundary (x=3.5)')

# Arrows: upward for misclassified (weight ↑), downward for correct (weight ↓)
# After Round 1 the only misclassified point was x=2
ax.annotate('', xy=(2, 0.15), xytext=(2, 0.05),
            arrowprops=dict(arrowstyle='->', color='red', lw=2))
ax.text(2.1, 0.12, 'weight ↑', color='red', fontsize=9)

for xi in [1, 3, 4, 5]:
    ax.annotate('', xy=(xi, -0.15), xytext=(xi, -0.05),
                arrowprops=dict(arrowstyle='->', color='blue', lw=1.5))
ax.text(3.2, -0.22, 'weight ↓ (correct)', color='blue', fontsize=9)

# Shade final ensemble decision (both learners agree on +1 for x>3)
ax.axvspan(3.5, 5.5, alpha=0.15, color='blue', label='Ensemble +1 region')
ax.axvspan(0.5, 1.5, alpha=0.15, color='red', label='Ensemble −1 region')

ax.set_xlim(0.5, 5.5)
ax.set_ylim(-0.4, 0.4)
ax.set_yticks([])
ax.set_xlabel('x')
ax.set_title('AdaBoost weight redistribution after Round 1\n(circle size ∝ sample weight)')
ax.legend(loc='upper left', fontsize=8)
ax.axhline(0, color='k', linewidth=0.5)

# ---------- Bottom panel: α bar chart ----------
ax2 = axes[1]
bars = ax2.bar(['Learner 1 (α₁)', 'Learner 2 (α₂)'], [alpha1, alpha2],
               color=['orange', 'green'], edgecolor='k')
ax2.set_ylabel('α (learner weight)')
ax2.set_title('Higher α ⇒ more trust in that weak learner')
for bar, val in zip(bars, [alpha1, alpha2]):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{val:.3f}', ha='center', fontsize=11)

plt.tight_layout()
plt.show()

print("\nPro tip reminder: A weight-redistribution diagram that shows samples")
print("'growing' or 'shrinking' across rounds explains AdaBoost more clearly")
print("than any equation. That is the visual interviewers and professors remember.")

---
## Q.4 — Explain It Cold

### 01 What is the key difference between AdaBoost and simple averaging?

**Why does AdaBoost assign a different $\alpha$ to each learner?**

**Answer:**  
Simple averaging treats every weak learner equally ($\alpha_t = 1$ or $1/T$).  
AdaBoost assigns a weight $\alpha_t = \frac12\ln\frac{1-\varepsilon_t}{\varepsilon_t}$ that is a **monotonic decreasing function of the weighted error $\varepsilon_t$**.  

- Accurate learners ($\varepsilon$ small) receive large positive $\alpha$ → they dominate the vote.  
- Poor learners ($\varepsilon$ close to 0.5) receive $\alpha$ near 0 → they contribute almost nothing.  
- Learners worse than random ($\varepsilon > 0.5$) receive negative $\alpha$ → their predictions are inverted.

This adaptive weighting is the core of AdaBoost’s power.

### 02 What happens to the ensemble if a weak learner has $\varepsilon = 0.5$?

Compute $\alpha$ for that case and interpret the result.

In [ ]:
eps = 0.5
alpha = 0.5 * math.log((1 - eps) / eps)
print(f"α = ½ ln((1-0.5)/0.5) = ½ ln(1) = {alpha}")

**Answer:**  
$\alpha = 0$.  
The learner is given zero weight and is effectively ignored by the ensemble.  
It contributes nothing to the final prediction.

### 03 How does AdaBoost handle regression differently from classification?

What changes in the loss function and aggregation?

**Answer:**  

| Aspect              | Classification (AdaBoost)                          | Regression (e.g. AdaBoost.R2)                          |
|---------------------|----------------------------------------------------|--------------------------------------------------------|
| Loss / error        | 0-1 misclassification (weighted)                   | Continuous residual (often squared or absolute)        |
| Weak learner output | $\pm 1$                                            | Real-valued prediction                                 |
| Learner weight $\alpha$ | $\frac12\ln\frac{1-\varepsilon}{\varepsilon}$     | Function of the relative loss (different formula)      |
| Aggregation         | Weighted majority vote $\operatorname{sign}(\sum\alpha_t h_t)$ | Weighted median (or weighted average) of the $h_t$ values |
| Sample weight update| Multiplicative by $e^{\pm\alpha}$                  | Based on the magnitude of residual                     |

The key conceptual change is moving from a discrete 0-1 loss to a continuous loss, and from a sign-of-sum vote to a weighted median/average of real numbers.

### 04 Why must each base learner be only slightly better than random?

What breaks if a learner is too strong?

**Answer:**  
AdaBoost’s theoretical guarantee (and its practical robustness) rests on the weak-learning assumption: every base learner satisfies $\varepsilon_t < 0.5$ (i.e., better than random guessing under the current distribution).

If a base learner is **too strong** (very low $\varepsilon$):

1. Its $\alpha$ becomes extremely large.
2. After the weight update the remaining samples that were still misclassified receive enormous relative weight.
3. Subsequent learners are forced to focus on a tiny, possibly noisy, subset of the data → overfitting.
4. The ensemble can become unstable; a single strong learner can dominate and the boosting process loses its ability to correct residual errors gradually.

In the extreme case $\varepsilon=0$ the algorithm formally stops (or $\alpha\to\infty$), which is undesirable when the “perfect” fit is actually fitting noise.

Hence AdaBoost works best when each weak learner is only **marginally better than random** — enough to make progress, but not so strong that it collapses the weight distribution or overfits.

---
## Summary of Key Numerical Results (realistic setting)

| Quantity | Value |
|----------|-------|
| Initial weights | $[0.2, 0.2, 0.2, 0.2, 0.2]$ |
| $\varepsilon_1$ (realistic) | $0.2$ |
| $\alpha_1$ | $\approx 0.693$ |
| Updated weights $w^{(2)}$ | $\approx [0.125, 0.500, 0.125, 0.125, 0.125]$ |
| $\varepsilon_2$ | $0.125$ |
| $\alpha_2$ | $\approx 0.973$ |
| Final prediction for $x=4$ | $+1$ |